In [1]:
import os
from dotenv import load_dotenv

# Load the environment variables and override any existing ones in the kernel
load_dotenv(override=True)

# Quick check to ensure the notebook found your .env file!
print(f"GOOGLE_API_KEY loaded: {bool(os.getenv('GOOGLE_API_KEY'))}")
print(f"TAVILY_API_KEY loaded: {bool(os.getenv('TAVILY_API_KEY'))}")

GOOGLE_API_KEY loaded: True
TAVILY_API_KEY loaded: True


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults

# 1. Initialize the Flash Lite model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

# 2. Initialize the Web Search Tool
web_search_tool = TavilySearchResults(max_results=3)

# 3. Bind the tool to the LLM
llm_with_tools = llm.bind_tools([web_search_tool])

print("✅ Model and Tools initialized successfully!")

✅ Model and Tools initialized successfully!


/var/folders/qj/pnj1tcsd4q3gxpt050rznh4m0000gn/T/ipykernel_6265/1589772547.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
/var/folders/qj/pnj1tcsd4q3gxpt050rznh4m0000gn/T/ipykernel_6265/1589772547.py:11: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(max_results=3)


In [3]:
from langchain_core.messages import HumanMessage

# Define test variables
target_sector = "Heavy Industry"
target_region = "India"

# Write the prompt
prompt = (
    f"You are a compliance automation bot. Find the official 2026 environmental emission "
    f"limits, thresholds, or caps for the {target_sector} sector in {target_region}. "
    f"Use your search tool to fetch real-time data. Summarize the exact numeric limits "
    f"and provide the legal citation source."
)

print(f"🔍 AI is deciding whether to search the web for {target_sector} in {target_region}...")
response = llm_with_tools.invoke([HumanMessage(content=prompt)])

print("\n📋 RAW AI RESPONSE:")
print("-" * 50)
# Check if the AI decided to use the web search tool
if response.tool_calls:
    print(f"🛠️ The AI chose to use the following tool: {response.tool_calls[0]['name']}")
    print(f"🔎 Search Query it generated: {response.tool_calls[0]['args']}")
else:
    print(response.content)

🔍 AI is deciding whether to search the web for Heavy Industry in India...

📋 RAW AI RESPONSE:
--------------------------------------------------
🛠️ The AI chose to use the following tool: tavily_search_results_json
🔎 Search Query it generated: {'query': 'official 2026 environmental emission limits heavy industry India'}


In [4]:
from langchain_core.messages import ToolMessage

# 1. Get the tool call from Gemini's response
tool_call = response.tool_calls[0]

# 2. Manually execute the tool using the arguments Gemini picked
print("🌐 Executing live Tavily web search...")
tool_output = web_search_tool.invoke(tool_call["args"])

# 3. Pass the tool results back to Gemini so it can read them
print("🧠 Feeding search results back to Gemini for final summary...\n")
final_response = llm_with_tools.invoke([
    HumanMessage(content=prompt), 
    response, 
    ToolMessage(content=str(tool_output), tool_call_id=tool_call["id"])
])

print("📋 FINAL COMPLIANCE DATA:")
print("-" * 50)
print(final_response.content)
print("-" * 50)

🌐 Executing live Tavily web search...
🧠 Feeding search results back to Gemini for final summary...

📋 FINAL COMPLIANCE DATA:
--------------------------------------------------
[{'type': 'text', 'text': 'As a compliance automation bot, I have analyzed the current regulatory landscape regarding environmental emission limits for the heavy industry sector in India for 2026.\n\n### Summary of Regulatory Status\nAs of early 2026, India has transitioned from purely concentration-based emission standards (managed by the Central Pollution Control Board - CPCB) to a **compliance-based carbon market framework** focused on **Greenhouse Gas (GHG) Emission Intensity targets**.\n\nThere is no single "universal" numeric cap for all heavy industries. Instead, the government has moved toward **entity-specific intensity targets** based on baselines established for individual industrial units.\n\n### Key Findings\n*   **Shift to Intensity Targets:** In January 2026, the Government of India notified mandat

In [ ]:
# agent 2

In [7]:
import os
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage, ToolMessage

# 1. Broadened schema to capture contact info AND environmental metrics
class FullCorporateProfile(BaseModel):
    discovered_company: str = Field(description="Name of the company found in the target sector and region.")
    company_emission_metric: str = Field(description="The exact current GHG emission intensity or CO2 metrics found for this company (e.g., '2.3 tCO2e/tcs'). Use 'N/A' if unknown.")
    cso_name: str = Field(description="Full name of the Chief Sustainability Officer or ESG Head. Use 'N/A' if unknown.")
    designation: str = Field(description="Official corporate title/designation.")
    email: str = Field(description="Direct corporate email address or official ESG contact point. Use 'N/A' if missing.")
    discovery_context: str = Field(description="Brief notes on what was discovered about their emissions and leadership.")

# 2. Initialize Gemini and Tavily Search Tool
llm_agent2 = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
structured_llm_agent2 = llm_agent2.with_structured_output(FullCorporateProfile)
web_search_tool_agent2 = TavilySearchResults(max_results=3)
llm_with_tools_agent2 = llm_agent2.bind_tools([web_search_tool_agent2])

target_sector = "Heavy Industry"
target_region = "India"

# 3. Comprehensive prompt instructing Agent 2 to dig for both contacts and metrics
prompt_text = (
    f"You are a comprehensive corporate researcher. Your goal is to build a profile for a prospect.\n"
    f"1. Find a major prominent company in the '{target_sector}' sector operating in '{target_region}'.\n"
    f"2. Search the web to find that specific company's latest reported environmental metrics, specifically their GHG emission intensity or CO2 per tonne of production.\n"
    f"3. Search the web to find their Chief Sustainability Officer (CSO) or Head of ESG name and contact email.\n"
    f"4. Loop through as many web searches as needed until you have gathered both pieces of information, then compile them into the structured schema."
)

print(f"🕵️ Agent 2 is starting its comprehensive multi-hop investigation...\n")

messages = [HumanMessage(content=prompt_text)]

# 4. The Multi-Hop Agent Loop
while True:
    response = llm_with_tools_agent2.invoke(messages)
    messages.append(response)
    
    if not response.tool_calls:
        print("🧠 Agent 2 has concluded all web searches.")
        break 
        
    for tool_call in response.tool_calls:
        print(f"🛠️ Executing Search: {tool_call['args']}")
        tool_output = web_search_tool_agent2.invoke(tool_call["args"])
        messages.append(ToolMessage(content=str(tool_output), tool_call_id=tool_call["id"]))

print("\n🌐 Compiling all metrics and contact data into the final schema...")

# 5. Extract the complete structured corporate profile
final_corporate_profile = structured_llm_agent2.invoke(messages)

print("\n📊 COMPREHENSIVE CORPORATE INVESTIGATION RESULT:")
print("-" * 50)
print(final_corporate_profile.model_dump_json(indent=4))
print("-" * 50)
print("✅ Agent 2 multi-task data collection complete!")

🕵️ Agent 2 is starting its comprehensive multi-hop investigation...

🛠️ Executing Search: {'query': 'major prominent companies in Heavy Industry sector in India'}
🛠️ Executing Search: {'query': 'Tata Steel India environmental metrics GHG emission intensity CO2 per tonne production'}
🛠️ Executing Search: {'query': 'Tata Steel India Chief Sustainability Officer or Head of ESG name and contact email'}
🧠 Agent 2 has concluded all web searches.

🌐 Compiling all metrics and contact data into the final schema...

📊 COMPREHENSIVE CORPORATE INVESTIGATION RESULT:
--------------------------------------------------
{
    "discovered_company": "Tata Steel",
    "company_emission_metric": "1.85 tCO2/tcs (target by 2030), 2.34 tCO2/tcs (BF-BOF route, 2024)",
    "cso_name": "Saurabh Kundu",
    "designation": "Chief Corporate Sustainability",
    "email": "N/A",
    "discovery_context": "Tata Steel is a major player in India's heavy industry sector. The company has set a target to achieve 1.85 tons o

In [9]:
import os
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Define the final structured schema for the Auditor
class ComplianceVerdict(BaseModel):
    regulatory_limit_found: str = Field(description="The numeric legal limit or framework rules extracted from the law.")
    reported_metric_found: str = Field(description="The company's current reported metric.")
    compliance_status: str = Field(description="Strictly 'COMPLIANT' or 'NON-COMPLIANT'.")
    gap_calculation: str = Field(description="The mathematical difference between current reality and the law. State 'N/A' if math isn't possible.")
    verdict_reasoning: str = Field(description="A brief explanation analyzing current performance against the mandate, ignoring future targets.")

# 2. Initialize the LLM
llm_agent3 = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)
structured_auditor = llm_agent3.with_structured_output(ComplianceVerdict)

# 3. DYNAMICALLY pull data from your actual previous notebook variables!
laws_input = final_response.content
company_metrics_input = final_corporate_profile.company_emission_metric
company_name_input = final_corporate_profile.discovered_company

# 4. The Auditor Prompt
prompt_text = (
    "You are a strict corporate compliance auditor.\n"
    f"Target Company: {company_name_input}\n"
    f"Regulatory Framework found by Agent 1: {laws_input}\n"
    f"Company Metrics found by Agent 2: {company_metrics_input}\n\n"
    "Task:\n"
    "1. Compare the company's current performance against the regulatory limits or guidelines.\n"
    "2. Determine if they are currently COMPLIANT or NON-COMPLIANT (ignore their future targets like 2030; look at current data).\n"
    "3. Estimate the gap and write a concise audit verdict."
)

print(f"⚖️ Agent 3 is auditing {company_name_input} using the live outputs from Agents 1 & 2...\n")

# 5. Run the auditor
final_verdict = structured_auditor.invoke(prompt_text)

print("🚨 OFFICIAL COMPLIANCE VERDICT (LIVE DATA):")
print("-" * 50)
print(final_verdict.model_dump_json(indent=4))
print("-" * 50)
print("✅ Agent 3 gap analysis complete!")

⚖️ Agent 3 is auditing Tata Steel using the live outputs from Agents 1 & 2...

🚨 OFFICIAL COMPLIANCE VERDICT (LIVE DATA):
--------------------------------------------------
{
    "regulatory_limit_found": "Entity-specific GHG Emission Intensity targets under the Indian Carbon Market (ICM) framework, administered by BEE and MoEFCC.",
    "reported_metric_found": "2.34 tCO2/tcs (BF-BOF route, 2024)",
    "compliance_status": "NON-COMPLIANT",
    "gap_calculation": "N/A",
    "verdict_reasoning": "Tata Steel's reported metric of 2.34 tCO2/tcs for 2024 is a current operational figure. The regulatory framework mandates entity-specific GHG Emission Intensity targets, which are not provided in the current data. Without knowing the specific baseline and target assigned to Tata Steel for the 2024 compliance period, a direct comparison and gap calculation is not possible. However, the company's mention of a 2030 target suggests they may not yet be meeting current (unspecified) intensity requirem

In [10]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Initialize the LLM (No tools or complex schemas needed here, just creative text generation)
llm_agent4 = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.7 # A slightly higher temperature allows for more natural, persuasive writing
)

# 2. Dynamically pull in variables from ALL previous notebook cells
company_name = final_corporate_profile.discovered_company
cso_name = final_corporate_profile.cso_name
cso_title = final_corporate_profile.designation
framework_rules = final_verdict.regulatory_limit_found
audit_verdict = final_verdict.compliance_status
audit_reasoning = final_verdict.verdict_reasoning

# 3. Create the copywriter prompt
prompt_text = (
    "You are an expert enterprise sales copywriter specializing in ESG and corporate compliance software.\n"
    "Your goal is to draft a highly professional, cold outreach email to a corporate executive.\n\n"
    "Context Given:\n"
    f"- Target Company: {company_name}\n"
    f"- Recipient Name: {cso_name}\n"
    f"- Recipient Title: {cso_title}\n"
    f"- New Regulation: {framework_rules}\n"
    f"- Audit Status: {audit_verdict}\n"
    f"- Audit Rationale: {audit_reasoning}\n\n"
    "Guidelines:\n"
    "1. Keep it professional, consultative, and respectful. Do not sound alarmist or like you are accusing them.\n"
    "2. Explicitly reference the newly transitioning Indian Carbon Market (ICM) intensity targets.\n"
    "3. Highlight that tracking unit-specific baselines dynamically can be challenging.\n"
    "4. Since direct email was N/A, add a small placeholder note at the top indicating this should be sent via LinkedIn message if needed.\n"
    "5. Pitch an introductory call to showcase how our automated compliance platform handles dynamic intensity forecasting."
)

print(f"✉️ Agent 4 is drafting a hyper-personalized outreach sequence for {cso_name}...\n")

# 4. Generate the draft
outreach_draft = llm_agent4.invoke(prompt_text)

print("📝 FINAL OUTREACH COLD DRAFT:")
print("-" * 50)
print(outreach_draft.content)
print("-" * 50)
print("✅ Agent 4 compilation complete! The full notebook prototype is successful.")

✉️ Agent 4 is drafting a hyper-personalized outreach sequence for Saurabh Kundu...

📝 FINAL OUTREACH COLD DRAFT:
--------------------------------------------------
Subject: Navigating New Entity-Specific GHG Intensity Targets in the Indian Carbon Market

**(Note: If direct email is not possible, this message should be adapted for a LinkedIn InMail.)**

Dear Mr. Kundu,

I hope this message finds you well.

As the Chief Corporate Sustainability at Tata Steel, I understand your critical role in navigating the evolving landscape of environmental regulations. I am reaching out today to discuss the implications of the recently introduced entity-specific Greenhouse Gas (GHG) Emission Intensity targets under the Indian Carbon Market (ICM) framework, administered by the Bureau of Energy Efficiency (BEE) and the Ministry of Environment, Forest and Climate Change (MoEFCC).

We recognize that dynamically tracking and managing unit-specific baselines and targets can present significant operational 